# YTAgent — Free 3D Video + Kokoro Narration Worker

This notebook runs Wan 2.1 on Kaggle's free GPU. It connects OUTBOUND to YTAgent over HTTPS, so no Cloudflare tunnel or changing public URL is needed. Turn **Internet on** and select a **GPU** accelerator in Notebook settings. Keep the notebook running while a video is being generated.The worker uses Wan 2.1 for real 3D video scenes and Kokoro 82M for local documentary narration. Both run without paid generation APIs. Turn **Internet on** and select a **GPU** accelerator. Keep the notebook running while a film is being generated.

In [ ]:
import os, subprocess
repo_dir = "/kaggle/working/YTAgent"
if not os.path.isdir(repo_dir):
    subprocess.run(["git", "clone", "https://github.com/qingremzy/YTAgent.git", repo_dir], check=True)
else:
    subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(repo_dir + "/gpu-worker")
print("Installing worker packages (Python 3.13-compatible pins)...")
subprocess.run(["python", "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print("Installing Kokoro + Misaki source without the incompatible curated-transformers extra...")
subprocess.run(["python", "-m", "pip", "install", "--no-deps", "git+https://github.com/hexgrad/misaki.git"], check=True)
subprocess.run(["python", "-m", "pip", "install", "--no-deps", "git+https://github.com/hexgrad/kokoro.git"], check=True)
print("Installing espeak-ng for Kokoro pronunciation...")
subprocess.run(["apt-get", "-qq", "-y", "update"], check=True)
subprocess.run(["apt-get", "-qq", "-y", "install", "espeak-ng"], check=True)
print("Checking TTS imports...")
subprocess.run(["python", "-c", "from kokoro import KPipeline; import spacy; print('Kokoro and spaCy imports OK', spacy.__version__)"], check=True)
print("Setup complete. No tunnel software is needed.")

In [ ]:
import os
worker_token = os.environ.get("YT_AGENT_WORKER_TOKEN")
if not worker_token:
    try:
        from kaggle_secrets import UserSecretsClient
        worker_token = UserSecretsClient().get_secret("YT_AGENT_WORKER_TOKEN")
        os.environ["YT_AGENT_WORKER_TOKEN"] = worker_token
    except Exception:
        raise RuntimeError("The worker token is not available. Preserve the existing live notebook session, or add YT_AGENT_WORKER_TOKEN under Kaggle Notebook Add-ons → Secrets.")
print("Private worker token loaded. It will not be displayed.")

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected. In Kaggle Notebook settings, select a GPU accelerator.")

In [ ]:
import os, subprocess, sys
os.environ["YT_AGENT_SERVER_URL"] = "https://ytagent-production-1038.up.railway.app"
print("Starting one-job Kaggle GPU runner. It waits for the queued job, generates it, uploads scenes and narration, then exits.")
subprocess.run([sys.executable, "app.py", "--run-once", "--wait-seconds", "1200"], check=True)
print("YTAgent GPU job finished.")

In [ ]:
print("Kaggle batch worker cell completed. Check the YTAgent dashboard for the final video status.")

# Deprecated: Cloudflare tunnel is not used. The worker polls Railway over outbound HTTPS.

## How the free GPU worker runs

YTAgent launches this notebook as a Kaggle batch run only after a video job is queued. The worker claims one job, generates all ten Wan scenes and Kokoro narration tracks, uploads them to Railway, and exits. No always-on interactive session or Cloudflare tunnel is required.

Run cells 1–4 in order only when testing manually. In normal use, YTAgent starts the saved notebook automatically when the GPU job is ready.

In [ ]:
print("The one-job GPU runner reports its result to YTAgent. Check the studio's generation status for progress and completion.")

In [ ]:
import os, pathlib, torch
print("python_cwd=", os.getcwd())
print("repo_exists=", pathlib.Path('/kaggle/working/YTAgent').exists())
print("worker_dir_exists=", pathlib.Path('/kaggle/working/YTAgent/gpu-worker').exists())
print("app_exists=", pathlib.Path('/kaggle/working/YTAgent/gpu-worker/app.py').exists())
print("token_loaded=", bool(os.environ.get("YT_AGENT_WORKER_TOKEN")))
print("server_url_loaded=", bool(os.environ.get("YT_AGENT_SERVER_URL")))
print("gpu_available=", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu_name=", torch.cuda.get_device_name(0))